# 7장. RAG 시스템 평가 — 01. RAGAS 합성 테스트 데이터셋 생성

## RAG 시스템의 품질을 어떻게 측정하고 개선할까요?

RAG(Retrieval-Augmented Generation) 시스템을 구축했다면 이제 "잘 동작하는지" 확인해야 해요. 그런데 평가하려면 **테스트 데이터셋**이 먼저 필요합니다. 수백 개의 질문-답변 쌍을 손으로 작성하는 건 현실적이지 않죠.

이 노트북에서는 **RAGAS(Retrieval Augmented Generation Assessment)**를 활용해 문서에서 자동으로 테스트 데이터셋을 생성하는 방법을 다뤄요.

### 학습 목표

- RAGAS의 합성 데이터 생성 파이프라인 이해
- `TestsetGenerator`로 다양한 유형의 질문 자동 생성
- 생성된 데이터셋을 저장하고 다음 평가 단계에서 활용

### 사전 지식

- 6장 RAG 시스템 구축 (문서 로딩, 청킹, 벡터 검색)
- `langchain_community`의 문서 로더 사용법

### 7장 전체 구성

```
01. 테스트 데이터셋 생성 (RAGAS)     ← 지금 여기
02. RAGAS 4가지 지표로 RAG 평가
03. LangSmith 데이터셋과 LLM-as-Judge
04. 커스텀 평가자 구현
05. Heuristic 평가 (ROUGE, BLEU, METEOR)
06. Groundedness(근거성) 평가
07. 모델 비교 평가
08. 온라인 평가 (Online Evaluation)
```

## 합성 데이터 생성의 장점

수동 작성 대신 LLM으로 테스트 데이터를 만들면 어떤 이점이 있을까요?

- **시간 절약**: 수동 작성 대비 대부분의 시간을 절감
- **다양성**: 단순/추론/다중 컨텍스트 등 다양한 난이도의 질문 자동 생성
- **일관성**: 동일한 기준으로 균일한 품질의 데이터셋 구축
- **확장성**: 문서가 바뀌어도 즉시 새 데이터셋 생성 가능

### RAGAS 데이터 생성 흐름

```mermaid
flowchart TD
    A[원본 문서<br/>attention_paper.pdf] --> B[문서 로드<br/>LangChain Document Loaders]
    B --> C[문서 청킹<br/>RecursiveCharacterTextSplitter]
    C --> D[TestsetGenerator<br/>generate_with_chunks]
    D --> E[LLM 기반 생성<br/>질문 / 정답 / 컨텍스트]
    E --> F[테스트셋<br/>user_input / reference / reference_contexts]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C,D,E process
    class F output
```

> **강의 포인트**: 합성 데이터 생성은 RAG 평가의 진입 장벽을 낮추는 핵심 기술입니다. "테스트 데이터가 없어서 평가를 못 한다"는 변명이 사라집니다. `TestsetGenerator`에 LLM과 Embeddings를 전달하고 `generate_with_chunks()`를 호출하면 질문 유형 분배까지 자동 처리됩니다.

---

## 환경 설정

먼저 필요한 패키지를 설치하고 API 키를 설정해요.

In [1]:
# 필요한 패키지 설치
# !pip install -qU langchain ragas langchain-openai langchain-community pdfplumber langchain-text-splitters

In [2]:
# DeprecationWarning 억제 (RAGAS 내부 경고)
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)

import langchain
import ragas

print(f"LangChain Version: {langchain.__version__}")
print(f"RAGAS Version: {ragas.__version__}")

LangChain Version: 0.3.28
RAGAS Version: 0.4.3


In [3]:
from dotenv import load_dotenv
import os

# 환경변수 로드
load_dotenv()

# LangSmith 프로젝트 설정
os.environ["LANGSMITH_PROJECT"] = "01-Eval-RAGAS-DatasetGen"

# ---------------------------------------------------
# [LangSmith UI 확인 방법]
# 1. https://smith.langchain.com 접속
# 2. 좌측 Projects 클릭
# 3. "Eval-RAGAS-DatasetGen" 프로젝트 선택
# 4. 주안점: Datasets 메뉴 → 생성된 테스트 데이터셋 확인
# ---------------------------------------------------

---

## 문서 준비

이 실습에서는 "Attention is All You Need" 논문을 사용해 합성 데이터셋을 생성합니다. Transformer 아키텍처를 소개한 논문으로, 기술적 질문을 생성하기에 내용이 풍부해요.

In [4]:
import os

# ---------------------------------------------------
# 데이터 파일 경로 확인
# ---------------------------------------------------
file_path = "data/attention_paper.pdf"
if not os.path.exists(file_path):
    raise FileNotFoundError(f"파일을 찾을 수 없습니다: {file_path}\n현재 디렉토리: {os.getcwd()}")

In [5]:
# ---------------------------------------------------
# PDF 문서 로드 및 페이지 수 확인
# ---------------------------------------------------
from langchain_community.document_loaders import PDFPlumberLoader

# PDFPlumberLoader: 표, 레이아웃 정보까지 추출하는 PDF 로더
loader = PDFPlumberLoader("data/attention_paper.pdf")

# 문서 로딩 — 각 페이지가 Document 객체 1개로 변환됨
docs = loader.load()

print(f"총 페이지 수: {len(docs)}")

총 페이지 수: 15


In [6]:
# 첫 번째 페이지 내용 확인
print(f"첫 페이지 길이: {len(docs[0].page_content)} 문자")
print(f"\n내용 미리보기:")
print(docs[0].page_content[:300])


첫 페이지 길이: 2581 문자

내용 미리보기:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog


### 문서 청킹

`generate_with_chunks()` 메서드는 미리 분할한 **텍스트 청크 리스트**를 입력으로 받아요. 즉, 청킹 전략(chunk_size, overlap, 분할 방식)을 사용자가 직접 제어할 수 있습니다.

**내부 동작 흐름:**

1. 전달된 텍스트 청크 → 내부 `DocumentStore`에 자동 저장
2. `KeyphraseExtractor`가 각 청크에서 핵심 구문 추출
3. `EmbeddingExtractor`가 청크별 벡터 임베딩 생성
4. 청크 간 유사도(Cosine Similarity)와 중첩도(Overlap Score)를 계산하여 관련 청크 그룹핑

> **왜 직접 청킹하나요?** 청킹 전략이 생성되는 질문의 품질을 좌우하기 때문입니다. chunk_size가 너무 작으면 컨텍스트가 부족하고, 너무 크면 핵심 구문 추출이 흐려져요.

In [7]:
# 메타데이터 확인
print("현재 메타데이터:")
print(docs[0].metadata)

현재 메타데이터:
{'source': 'data/attention_paper.pdf', 'file_path': 'data/attention_paper.pdf', 'page': 0, 'total_pages': 15, 'Author': '', 'CreationDate': 'D:20240410211143Z', 'Creator': 'LaTeX with hyperref', 'Keywords': '', 'ModDate': 'D:20240410211143Z', 'PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'Producer': 'pdfTeX-1.40.25', 'Subject': '', 'Title': '', 'Trapped': 'False'}


> **실무 팁**: `generate_with_chunks()`는 순수 텍스트 리스트를 받으므로 metadata 설정이 필요 없어요. LangChain의 `RecursiveCharacterTextSplitter`로 청킹하되, 처음 8페이지만 사용해 실습 시간을 줄입니다.

In [8]:
# ---------------------------------------------------
# 문서 청킹: RecursiveCharacterTextSplitter로 분할
# ---------------------------------------------------
# ============================================================
# TODO: RecursiveCharacterTextSplitter로 문서를 청킹하세요
# 힌트: chunk_size=1500, chunk_overlap=200
# 예상 결과: 청크 수와 첫 번째 청크 길이 출력
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter

# 문서 분할기 생성
splitter = RecursiveCharacterTextSplitter(chunk_size=1500, chunk_overlap=200)

# 처음 8페이지만 사용 (전체 문서는 시간이 오래 걸림)
chunks = splitter.split_documents(docs[:8])

# generate_with_chunks()는 텍스트 리스트를 받으므로 page_content만 추출
text_chunks = [c.page_content for c in chunks]

print(f"청크 수: {len(text_chunks)}")
print(f"첫 번째 청크 길이: {len(text_chunks[0])} 문자")
print(f"\n첫 번째 청크 미리보기:")
print(text_chunks[0][:300])

청크 수: 20
첫 번째 청크 길이: 1499 문자

첫 번째 청크 미리보기:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog


> **핵심 개념**: `TestsetGenerator`는 하나의 LLM으로 질문 생성, 답변 생성, 품질 검증을 모두 처리합니다. Embeddings는 청크 간 유사도 계산과 관련 컨텍스트 선택에 사용돼요. Embeddings는 **RAGAS 네이티브** (`ragas.embeddings.OpenAIEmbeddings`)를 사용해야 합니다.

---

## RAGAS 테스트셋 생성기 설정

테스트셋을 만들기 위해 LLM과 Embeddings 두 가지만 준비하면 됩니다.

### 1단계: 필요한 컴포넌트 초기화

테스트셋 생성을 위해 다음 두 컴포넌트가 필요해요.

| 컴포넌트 | 역할 | 사용 클래스 |
|---------|------|-----------|
| **LLM** | 질문·답변 생성 + 품질 검증 | `LangchainLLMWrapper(ChatOpenAI(...))` |
| **Embeddings** | 청크 벡터화, 유사도 계산, 관련 컨텍스트 선택 | `ragas.embeddings.OpenAIEmbeddings` (RAGAS 네이티브) |

> **주의**: Embeddings는 LangChain의 `OpenAIEmbeddings`가 아니라 **RAGAS 자체** `OpenAIEmbeddings`를 사용해야 해요. RAGAS 네이티브 Embeddings는 `openai.OpenAI()` 클라이언트를 직접 받습니다.

In [9]:
# ---------------------------------------------------
# RAGAS 핵심 컴포넌트 초기화 (LLM / Embeddings)
# ---------------------------------------------------
# ============================================================
# TODO: LLM과 Embeddings를 초기화하세요
# 힌트: LangchainLLMWrapper(ChatOpenAI(...))
#       ragas.embeddings.OpenAIEmbeddings(client=openai.OpenAI(), model='text-embedding-3-small')
# 예상 결과: 두 객체가 초기화되어 출력
# ============================================================

import openai
from langchain_openai import ChatOpenAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import OpenAIEmbeddings as RagasOpenAIEmbeddings

# LLM: LangChain ChatOpenAI를 RAGAS 래퍼로 감싸기
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini"))

# Embeddings: RAGAS 네이티브 OpenAIEmbeddings (openai 클라이언트 직접 전달)
openai_client = openai.OpenAI()
generator_embeddings = RagasOpenAIEmbeddings(
    client=openai_client,
    model="text-embedding-3-small"
)

print(f"LLM: {generator_llm}")
print(f"Embeddings: {generator_embeddings}")

LLM: LangchainLLMWrapper(langchain_llm=ChatOpenAI(...))
Embeddings: OpenAIEmbeddings(provider='openai', model='text-embedding-3-small', client=<OpenAI:sync>)


### 2단계: TestsetGenerator 내부 구조

`TestsetGenerator`는 생성자에 `llm`과 `embedding_model`만 전달하면 내부에서 다음 컴포넌트를 자동 초기화합니다.

```mermaid
flowchart TD
    A[TestsetGenerator] --> B[InMemoryDocumentStore<br/>청크 저장·관리]
    A --> C[KeyphraseExtractor<br/>핵심 구문 추출]
    A --> D[EmbeddingExtractor<br/>벡터 임베딩 생성]
    A --> E[ThemesExtractor<br/>주제 추출]
    A --> F[NERExtractor<br/>개체명 인식]
    
    B --> G[CosineSimilarityBuilder<br/>청크 간 유사도 계산]
    B --> H[OverlapScoreBuilder<br/>청크 간 중첩도 계산]
    
    G --> I[Persona 생성<br/>다양한 관점의 질문자 페르소나]
    H --> I
    I --> J[Scenario 생성<br/>질문 시나리오 구성]
    J --> K[Sample 생성<br/>최종 질문-답변 쌍]

    classDef auto fill:#e8f5e9,stroke:#4caf50,color:#1b5e20
    class B,C,D,E,F,G,H,I,J,K auto
```

> 사용자가 직접 초기화할 필요 없이, `TestsetGenerator` 생성자 호출 한 번으로 전체 파이프라인이 준비됩니다.

> **실무 팁**: 처음에는 소량(6~10개)으로 생성해 결과가 의도대로 나오는지 먼저 확인하세요. 문서 내용에 따라 질문 유형이 자동 분배되므로, 결과를 검토하며 `testset_size`를 조절하는 것이 좋습니다.

### 3단계: TestsetGenerator 생성

`TestsetGenerator` 생성자에 `llm`과 `embedding_model`을 직접 전달합니다.

In [10]:
# ---------------------------------------------------
# TestsetGenerator 초기화
# ---------------------------------------------------
from ragas.testset import TestsetGenerator

# TestsetGenerator: 생성자에 LLM + Embeddings 전달
generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)

print("TestsetGenerator 초기화 완료")

TestsetGenerator 초기화 완료


---

## 질문 유형별 분포

RAGAS는 문서 내용을 분석하여 질문 유형을 자동으로 분배합니다. 내부적으로 다음 3가지 Synthesizer가 사용돼요.

| Synthesizer | 설명 | 예시 |
|------------|------|------|
| `single_hop_specific_query_synthesizer` | 단일 청크에서 **구체적 사실**을 묻는 질문 | "X의 값은?", "Y는 무엇인가?" |
| `multi_hop_specific_query_synthesizer` | 여러 청크를 조합해 **구체적 답변**을 요구하는 질문 | "A와 B의 차이점은?" |
| `multi_hop_abstract_query_synthesizer` | 여러 청크를 종합해 **추상적 이해**를 묻는 질문 | "전체적인 접근 방식은?" |

### 자동 분배 원리

1. **Persona 생성**: 문서 주제에 맞는 가상 질문자 페르소나를 LLM이 생성
2. **Scenario 구성**: 각 페르소나가 할 법한 질문 시나리오를 설계
3. **유형 자동 선택**: 시나리오의 복잡도에 따라 single_hop / multi_hop 유형이 자동 결정
4. **균형 분배**: 최종적으로 유형별 균등 분배를 목표로 조정

---

## 테스트셋 생성

모든 설정이 완료되었어요. `generate_with_chunks()` 호출 시 내부에서 다음 순서로 처리돼요.

1. **SummaryExtractor** — 각 청크의 요약 생성
2. **EmbeddingExtractor** — 청크별 벡터 임베딩 계산
3. **ThemesExtractor / NERExtractor** — 주제와 개체명 추출
4. **CosineSimilarity / OverlapScore** — 청크 간 관계 그래프 구축
5. **Persona 생성** — 문서 주제 기반 가상 질문자 생성
6. **Scenario 생성** — 페르소나별 질문 시나리오 구성
7. **Sample 생성** — 최종 질문-답변-컨텍스트 트리플 생성

In [11]:
# ---------------------------------------------------
# 합성 테스트셋 생성
# ---------------------------------------------------
# ============================================================
# TODO: generate_with_chunks()를 호출해 테스트셋을 생성하세요
# 힌트: generator.generate_with_chunks(chunks=text_chunks, testset_size=10)
# 예상 결과: "테스트셋 생성 완료!" 출력
# ============================================================

# generate_with_chunks: 텍스트 리스트를 직접 전달
testset = generator.generate_with_chunks(
    chunks=text_chunks,    # 앞에서 만든 텍스트 청크 리스트
    testset_size=10,       # 생성할 질문 수
)

print("\n테스트셋 생성 완료!")

Applying SummaryExtractor:   0%|          | 0/20 [00:00<?, ?it/s]

Task failed with RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Task failed with RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}
Task failed with RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None,

RateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}

### 데이터셋 구조

생성된 데이터셋은 다음 컬럼을 포함해요.

| 컬럼 | 설명 | 다음 노트북에서 역할 |
|------|------|-------------------|
| `user_input` | 생성된 질문 | RAG 시스템에 입력 |
| `reference` | 참조 정답 | Context Recall 계산 기준 |
| `reference_contexts` | 관련 문서 청크 리스트 | Context Precision 계산 기준 |
| `synthesizer_name` | 질문 생성기 유형 | 유형별 성능 분석 |
| `persona_name` | 질문자 페르소나 이름 | 관점별 질문 다양성 확인 |
| `query_style` / `query_length` | 질문 스타일과 길이 | 질문 특성 분석 |

In [ ]:
# ---------------------------------------------------
# 생성된 테스트셋을 DataFrame으로 변환하여 구조 확인
# ---------------------------------------------------
test_df = testset.to_pandas()

print(f"생성된 테스트 케이스 수: {len(test_df)}")
print(f"\n컬럼: {list(test_df.columns)}")

if len(test_df) == 0:
    print("\n[주의] 테스트 케이스가 생성되지 않았습니다.")
else:
    test_df.head()

생성된 테스트 케이스 수: 12

컬럼: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


In [ ]:
# 질문 생성기 유형별 개수 확인
if len(test_df) > 0:
    if 'synthesizer_name' in test_df.columns:
        print("생성기 유형별 분포:")
        print(test_df['synthesizer_name'].value_counts())
        print(f"\n비율:")
        print(test_df['synthesizer_name'].value_counts(normalize=True).round(2))
else:
    print("[주의] 테스트 케이스가 생성되지 않아 분포를 표시할 수 없습니다.")

생성기 유형별 분포:
synthesizer_name
single_hop_specific_query_synthesizer    4
multi_hop_abstract_query_synthesizer     4
multi_hop_specific_query_synthesizer     4
Name: count, dtype: int64

비율:
synthesizer_name
single_hop_specific_query_synthesizer    0.33
multi_hop_abstract_query_synthesizer     0.33
multi_hop_specific_query_synthesizer     0.33
Name: proportion, dtype: float64


> **자주 하는 실수**: CSV로 저장하면 `reference_contexts` 컬럼의 Python 리스트가 문자열로 변환됩니다. 다음 노트북에서 불러올 때 `ast.literal_eval()`로 반드시 복원해야 합니다. 이를 빠뜨리면 RAGAS 평가 시 타입 오류가 발생합니다.

In [ ]:
# 첫 번째 테스트 케이스 상세 확인
if len(test_df) > 0:
    print("=" * 80)
    print("테스트 케이스 예시")
    print("=" * 80)

    sample = test_df.iloc[0]

    print(f"\n질문 (user_input):")
    print(sample['user_input'])

    print(f"\n정답 (reference):")
    print(sample.get('reference', 'N/A'))

    if 'reference_contexts' in test_df.columns and sample.get('reference_contexts') is not None:
        contexts = sample['reference_contexts']
        print(f"\n관련 컨텍스트 수: {len(contexts)}")
        print(f"\n컨텍스트 1:")
        print(str(contexts[0])[:300] + "...")

    print(f"\n생성기 유형 (synthesizer_name): {sample.get('synthesizer_name', 'N/A')}")

테스트 케이스 예시

질문 (user_input):
Wht is Google Resarch known for?

정답 (reference):
Google Research is known for its contributions to advanced machine learning techniques, including the development of the Transformer architecture, which is based solely on attention mechanisms and has shown superior performance in machine translation tasks.

관련 컨텍스트 수: 1

컨텍스트 1:
Providedproperattributionisprovided,Googleherebygrantspermissionto
reproducethetablesandfiguresinthispapersolelyforuseinjournalisticor
scholarlyworks.
Attention Is All You Need
AshishVaswani∗ NoamShazeer∗ NikiParmar∗ JakobUszkoreit∗
GoogleBrain GoogleBrain GoogleResearch GoogleResearch
avaswani@goog...

생성기 유형 (synthesizer_name): single_hop_specific_query_synthesizer


---

## 데이터셋 저장

In [ ]:
# ---------------------------------------------------
# 생성된 테스트셋을 CSV 파일로 저장
# ---------------------------------------------------
if len(test_df) > 0:
    output_path = "data/attention_synthetic_testset.csv"
    test_df.to_csv(output_path, index=False)
    print(f"테스트셋이 저장되었습니다: {output_path}")
    print(f"총 {len(test_df)}개의 테스트 케이스")
else:
    print("[주의] 테스트 케이스가 생성되지 않아 저장을 건너뜁니다.")
    output_path = None

테스트셋이 저장되었습니다: data/attention_synthetic_testset.csv
총 12개의 테스트 케이스


### 저장 데이터 검증

> **주의**: CSV로 저장하면 `reference_contexts` 컬럼의 리스트가 문자열로 변환돼요. 다음 노트북에서 `ast.literal_eval()`로 다시 리스트로 변환해야 합니다.

In [ ]:
import pandas as pd

if output_path:
    loaded_df = pd.read_csv(output_path)
    print(f"로드된 데이터: {len(loaded_df)} rows x {len(loaded_df.columns)} columns")
    print(f"\n컬럼: {list(loaded_df.columns)}")
    loaded_df.head(3)
else:
    print("저장된 파일이 없습니다.")

로드된 데이터: 12 rows x 7 columns

컬럼: ['user_input', 'reference_contexts', 'reference', 'persona_name', 'query_style', 'query_length', 'synthesizer_name']


---

## 정리

### RAGAS 합성 데이터 생성 요약

| 단계 | 작업 | 핵심 포인트 |
|------|------|-----------|
| 1. 문서 준비 | PDF 로드 | PDFPlumberLoader 사용 |
| 2. 문서 청킹 | RecursiveCharacterTextSplitter | chunk_size=1500, chunk_overlap=200 |
| 3. 컴포넌트 설정 | LLM + Embeddings 초기화 | LangchainLLMWrapper + RAGAS 네이티브 Embeddings |
| 4. Generator 생성 | `TestsetGenerator(llm=..., embedding_model=...)` | 내부 컴포넌트 자동 초기화 |
| 5. 테스트셋 생성 | `generate_with_chunks()` 호출 | 텍스트 리스트 전달, `testset_size`로 수량 지정 |
| 6. 저장 및 검증 | CSV 저장 + 컬럼 확인 | reference_contexts는 문자열로 저장됨 주의 |

### TestsetGenerator 내부 파이프라인

```
텍스트 청크 리스트
    → SummaryExtractor (요약 생성)
    → EmbeddingExtractor (벡터 임베딩)
    → ThemesExtractor + NERExtractor (주제·개체명 추출)
    → CosineSimilarity + OverlapScore (관계 그래프 구축)
    → Persona 생성 (가상 질문자)
    → Scenario 생성 (질문 시나리오)
    → Sample 생성 (질문-답변-컨텍스트 트리플)
```

### 다음 노트북 예고

생성한 테스트 데이터셋을 활용해서 RAG 시스템의 성능을 **RAGAS의 4가지 지표**로 평가해볼게요. Faithfulness(충실도), Answer Relevancy(답변 관련성), Context Precision(컨텍스트 정확도), Context Recall(컨텍스트 재현율)이 각각 무엇을 측정하는지 살펴봅니다.